# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability initialization** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [ ]:
# --- Run identity ---
protocol_mode = "execution"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move and the 96-head sections only run when this is True.
allow_x_arm_move = True

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the head's own initialization command in an earlier run.
head96_tip_discard_location = (-263.8, 108.3, 200.0)  # as a Coordinate below, after imports

# --- Tips ---
# Section 13 collects a rack of tips on the 96-head and puts them straight back. Off by default:
# tip handling is not what this driver is being validated for yet, and the section models a carrier
# onto the deck as soon as it runs, which would put the model out of step with a deck that has none.
# To use it, set this to the track a tip carrier is physically loaded on.
tip_carrier_track = None

## 2- Imports

In [ ]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.master import STARDriver
from pylabrobot.hamilton.star.resource_model import NChannelPipette, TipMountingShaft
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.hamilton import TIP_CAR_480_A00, hamilton_96_tiprack_1000uL

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [ ]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_tip_discard_location` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [ ]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# A capability cannot be configured before setup: it hangs off an arm, and the arms are not known
# until discovery has read the machine. So setup runs first and reports that it could not initialize
# the 96-head, and the cell below does that separately.
await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

In [ ]:
deck = star.deck
deck

In [ ]:
star.driver.deck

### Initializing the 96-head

**This descends.** The head ejects whatever is mounted when it initializes, at
`head96_tip_discard_location`, which is machine-specific and cannot be checked from here. Gated on
`allow_head96_initialize`, set in the cell itself.

Setup retracts the head either way, so everything below runs whether or not this does.

In [ ]:
allow_head96_initialize = False

# Initializing the 96-head makes it descend to `head96_tip_discard_location` and eject whatever is
# mounted, so it is separate from setup and off by default: where a head ejects is machine-specific
# and nothing here can check it. Setup leaves the head retracted either way, so the sections below
# work whether or not this runs - unless the head refuses to move while it reports itself
# uninitialized, which is what this is here for.
if star.head96 is None:
  print("no 96-head on this machine")
elif head96_tip_discard_location is None:
  print("no head96_tip_discard_location set at the top of the notebook")
elif await star.driver.request_initialization_status(star.head96.configuration.module):
  print("the 96-head reports itself initialized already; nothing to do")
elif not allow_head96_initialize:
  print(
    f"skipped. the head reports itself uninitialized. set allow_head96_initialize = True to send "
    f"it to {head96_tip_discard_location} and eject"
  )
else:
  star.head96.configuration.tip_discard_location = Coordinate(*head96_tip_discard_location)
  await star.head96.initialize()
  print(f"initialized; the head ejected at {head96_tip_discard_location}")

## 5- What setup found

Everything below depends on what discovery read, so this prints it in full before anything uses it.

The last table is the one to look at. The machine's configuration says what is installed; the driver
builds a capability for each of those. A row where the two disagree means discovery declined to
build something the machine claims - which is either a bug or a bit set on a machine that no longer
carries the part.

In [ ]:
print(star.driver.format_setup_summary())

c = star.driver.configuration
print(f"\nchannels    : {star.driver.num_channels}")
print(f"instrument  : {c.instrument_size_slots} slots")

print("\nfirmware, per module:")
for module, version in star.driver.firmware.items():
  print(f"  {module:<12} {version}")

# What the configuration claims, against what the driver built for it.
rows = [
  ("autoload", c.autoload_installed, star.driver.autoload),
  ("front cover", c.main_front_cover_monitoring_installed, star.front_cover),
]
for arm in star.driver.arms:
  a = arm.configuration
  rows += [
    (f"{arm.side} channels", a.pip_installed, arm.pipettes),
    (f"{arm.side} 96-head", a.head96_installed, arm.head96),
    (f"{arm.side} 384-head", a.head384_installed, arm.head384),
    (f"{arm.side} iSWAP", a.iswap_installed, arm.iswap),
  ]

print(f"\n  {'capability':<20} {'installed':<11} {'built'}")
for label, installed, built in rows:
  disagree = "   <- disagree" if bool(installed) != (built is not None) else ""
  print(
    f"  {label:<20} {str(bool(installed)):<11} {'yes' if built is not None else 'no'}{disagree}"
  )

## 6- The resource model

What the machine carries is modelled as resources on the deck, placed where the drives say they are:
each arm, every pipetting channel, each head, the autoload sled. Nothing here moves or reads the
machine - it prints what setup already built.

The tip mounting shafts are the part that matters for tips. Each pipetting channel and each head
channel carries one, and a shaft is what a collected tip is meant to hang from - so the tree itself
would answer what is mounted, with no separate record to keep in step with it.

`mount_tip` and `release_tip` are on the shaft, but nothing calls them yet: collecting tips does not
put them on the model. The count below therefore reads zero whatever the head is carrying, and
section 13 is where that shows.

In [ ]:
def show(resource, indent=0, max_depth=3, max_siblings=4):
  """Print a resource and its children, abbreviating wide and deep branches."""
  print(f"{'  ' * indent}{resource.name}  ({type(resource).__name__})")
  if indent >= max_depth:
    if resource.children:
      print(f"{'  ' * (indent + 1)}... {len(resource.children)} children")
    return
  for child in resource.children[:max_siblings]:
    show(child, indent + 1, max_depth, max_siblings)
  if len(resource.children) > max_siblings:
    print(f"{'  ' * (indent + 1)}... {len(resource.children) - max_siblings} more")


show(star)

children = star.get_all_children()
shafts = [r for r in children if isinstance(r, TipMountingShaft)]
print(f"\n{len(shafts)} tip mounting shafts, {sum(s.has_tip() for s in shafts)} carrying a tip")

for pipette in [r for r in children if isinstance(r, NChannelPipette)]:
  print(
    f"\n{pipette.name}: {pipette.num_channels} channels, {pipette.channel_pitch} mm pitch, "
    f"{pipette.tip_pickup_mode} pickup, independent actuation "
    f"{pipette.independent_channel_actuation}"
  )
  a1 = pipette.get_item("A1")
  print(f"  A1 at {a1.get_location_wrt(deck)} in deck mm")

## 7- Has this firmware stack been driven before?

What the machine is made of is captured in full by the capture script; what a run wants to know here
is whether any of its boards report firmware this driver has not been driven against.


In [ ]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))

## 8- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [ ]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

## 9- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [ ]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

## 10- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [ ]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

In [ ]:
# The same gate as the section above: this moves the arm. It reads the gate rather than setting
# it - setting it here would leave every later section unlocked too.
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped the move. the comparison below runs at wherever the arm is now")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

## 11- The 96-head: park

Park is two moves, not the head's own home command: up to safe Z first, then across in Y to the
first of the Y positions the head has stored. In that order because the head crosses the deck to get
there - raised it sweeps over what is loaded, low it sweeps through it.

Only the order has been checked, in simulation. This is the first time the machine runs it.

**This moves the 96-head** in Z and then in Y. The deck has to be clear along its sweep. Gated on
`allow_x_arm_move`.

In [ ]:
if not allow_x_arm_move:
  print("skipped. set allow_x_arm_move = True to move the head")
elif star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  print(
    f"before : y={await head.request_y_position():7.2f}  z={await head.request_z_position():7.2f}"
  )

  stored = await head.request_predefined_y_positions()
  print(f"stored y positions: {stored}, so park is {stored[0]:.2f} mm")

  parked_at = await head.park()
  y_now, z_now = await head.request_y_position(), await head.request_z_position()
  print(f"after  : y={y_now:7.2f}  z={z_now:7.2f}, park() reported {parked_at:.2f}")

  # The head is a resource too, and park moved it. The model should have followed.
  if head.resource is not None:
    print(f"model  : {head.resource.get_item('A1').get_location_wrt(deck)}")

## 12- The 96-head: the Y floor

This head refuses to travel as far forward as its Y parameter accepts. The rig settled where the
real limit is (section 17) and the driver now carries it, so what is left to check is that the guard
and the machine still agree.

Three probes. The driver should refuse a target below the floor without anything reaching the wire;
the machine should accept the floor itself; and the drive should report it arrived there.

**The second probe moves the 96-head** forward in Y, to the front of its permitted area. It is
raised to safe Z first. Gated on `allow_x_arm_move`.

In [ ]:
if star.head96 is None:
  print("no 96-head on this machine")
else:
  head = star.head96
  floor, ceiling = head.configuration.y_range
  print(f"permitted Y area: {floor:.3f} to {ceiling:.3f} mm")

  # 1- the driver refuses below the floor, before anything reaches the wire. No motion.
  try:
    await head.move_y(floor - 0.5)
    print(f"  {floor - 0.5:7.3f} mm  NOT REFUSED - the guard is not holding")
  except ValueError as error:
    print(f"  {floor - 0.5:7.3f} mm  refused by the driver: {error}")

  # 2- and the machine takes the floor itself.
  if not allow_x_arm_move:
    print("  skipped the move. set allow_x_arm_move = True to send the head to the floor")
  else:
    await head.move_to_safe_z()
    await head.move_y(floor)
    arrived = await head.request_y_position()
    # Against the drive's own repeatability, not against nothing: the position read is the hardware
    # counter, and it settles within about a tenth of a millimetre of what was commanded.
    settle = 0.15
    print(f"  {floor:7.3f} mm  accepted, the drive reports {arrived:7.3f} mm")
    print(f"  within {settle} mm of target: {abs(arrived - floor) <= settle}")
    await head.park()
    print(f"  parked at {await head.request_y_position():.2f} mm")

## 13- The 96-head: tips

The head collects a whole rack at once - it is rigid, so there is no per-channel selection - and this
picks a rack up and puts it straight back on the same rack. Nothing is thrown away.

Three checks, and one of them is expected to fail:

- **the commands** - the head should build the same collect and drop commands the legacy driver did
- **the sleeve sensors** - 96 channels should report a tip after the collect and none after the drop
- **the model** - the shafts should carry 96 tips, and they will not. Nothing calls `mount_tip` yet,
  so this reads zero both times. It is here to measure the gap, not to pass.

Where a discard would go is printed but not sent: the head's configured trash resolves when no
location is given, and that resolution is worth seeing without binning a rack of tips to see it.

**This moves the 96-head** in X, Y and Z, over the tip carrier. A tip carrier has to be loaded on
`tip_carrier_track`, with a full rack on its frontmost site. Gated on `allow_head96_tip_moves`, set
in the cell itself so running the notebook top to bottom does not reach for tips.

In [ ]:
allow_head96_tip_moves = True

if star.head96 is None:
  print("no 96-head on this machine")
elif tip_carrier_track is None:
  print("no tip_carrier_track set at the top of the notebook")
else:
  head = star.head96

  # The carrier as it is physically loaded, so the rack's own geometry decides where the head goes.
  if "tip_carrier" in [child.name for child in deck.children]:
    carrier = deck.get_resource("tip_carrier")
    rack = deck.get_resource("tip_rack")
  else:
    carrier = TIP_CAR_480_A00(name="tip_carrier")
    rack = hamilton_96_tiprack_1000uL(name="tip_rack")
    carrier[0] = rack
    deck.assign_child_resource(carrier, rails=tip_carrier_track)

  a1 = rack.get_item("A1").get_location_wrt(deck)
  floor, ceiling = head.configuration.y_range
  print(f"rack A1 sits at {a1}")
  print(f"the head's permitted Y area is {floor:.2f} to {ceiling:.2f} mm")
  if not floor <= a1.y <= ceiling:
    raise RuntimeError(
      f"the head cannot reach the rack: its A1 is at y={a1.y:.2f} mm, outside {floor:.2f} to "
      f"{ceiling:.2f}. Load the carrier further back, or use a site further back on it."
    )

  # Where a discard would go, resolved the way discard_tips resolves it. Nothing is sent.
  configured = head.configuration.tip_discard_location
  print(f"a discard with no location would go to {configured}")

  if not allow_head96_tip_moves:
    print("\nskipped the moves. set allow_head96_tip_moves = True to collect and return a rack")
  else:
    shafts = [r for r in star.get_all_children() if isinstance(r, TipMountingShaft)]

    await head.pick_up_tip_rack(rack)
    print(
      f"\ncollected. sensors: {sum(await star.driver.request_tip_presence())} channels report a tip"
    )
    print(
      f"           model  : {sum(s.has_tip() for s in shafts)} shafts carry one (expected 0 - not wired)"
    )
    print(f"           rack   : {sum(s.has_tip() for s in rack.get_all_items())} spots still full")

    await head.drop_tips(rack)
    print(
      f"\nreturned.  sensors: {sum(await star.driver.request_tip_presence())} channels report a tip"
    )
    print(
      f"           model  : {sum(s.has_tip() for s in shafts)} shafts carry one (expected 0 - not wired)"
    )
    print(f"           rack   : {sum(s.has_tip() for s in rack.get_all_items())} spots full again")

    await head.park()

## 14- The 384-head

Nothing on this machine, which is the point: the driver has to say so rather than assume a head that
is not there. On a machine that does carry one, this reads what it says about itself - all read-only,
nothing moves.

In [ ]:
head384 = star.head384
if head384 is None:
  installed = any(arm.configuration.head384_installed for arm in star.driver.arms)
  print(f"no 384-head built. any arm reports one installed: {installed}")
else:
  cfg = head384.configuration
  print(f"type              : {cfg.head_type}")
  print(
    f"channels          : {cfg.channel_columns} x {cfg.channel_rows}, {cfg.channel_pitch} mm pitch"
  )
  print(f"permitted Y area  : {cfg.y_range[0]:.2f} to {cfg.y_range[1]:.2f} mm")
  print(
    f"permitted Z area  : {cfg.z_range_documented[0]:.2f} to {cfg.z_range_documented[1]:.2f} mm"
  )
  # Both are the head type's to decide, so they refuse until it has been read.
  try:
    print(f"dispensing drive  : {cfg.dispensing_drive_uL_per_increment:.9f} uL per increment")
    print(f"squeezer drive    : {cfg.squeezer_drive_mm_per_increment:.9f} mm per increment")
  except RuntimeError as error:
    print(f"drive resolutions : {error}")
  print(f"absolute cLLD     : {cfg.supports_lld_absolute_threshold_check}")
  print(
    f"\nthe drives report: y={await head384.request_y_position():.2f}  "
    f"z={await head384.request_z_position():.2f}"
  )

## 15- The iSWAP

Read-only. Every value below is one the arm keeps and reports; none of it moves the gripper.

The two link lengths and the rotation drive's X offset are what a kinematic model of the arm needs,
and until now the driver took them on faith. This is the check that the arm answers them, and that
what it answers is the geometry we assumed.

In [ ]:
iswap = star.iswap
if iswap is None:
  installed = any(arm.configuration.iswap_installed for arm in star.driver.arms)
  print(f"no iSWAP built. any arm reports one installed: {installed}")
else:
  print(f"firmware          : {await iswap.request_firmware_version()}")
  print(f"link 1            : {await iswap.request_link_1_length():.2f} mm")
  print(f"link 2            : {await iswap.request_link_2_length():.2f} mm")
  print(f"rotation X offset : {await iswap.request_rotation_drive_x_offset():.2f} mm")
  print(f"rotation drive    : {await iswap.request_rotation_drive_positions()}")
  print(f"wrist drive       : {await iswap.request_wrist_drive_positions()}")
  print(f"Y positions       : {await iswap.request_y_positions()}")
  print(
    f"gripper           : {'wide' if star.driver.configuration.iswap_gripper_wide else 'small'}"
  )

## 16- Error decoding

An error comes back as a code on a module, and the driver turns it into a named exception with the
module that raised it and what it was doing. Two ways in, and both are checked here.

The guards first: a target outside a drive's range never reaches the wire at all, so the machine
never gets a chance to refuse it. Then a real one - a parameter the master does not know, which is
read-only and produces a genuine error to decode without moving anything.

In [ ]:
# 1- refused locally, nothing sent
try:
  await star.x_arm.move_x(5000.0)
except ValueError as error:
  print(f"local guard : {type(error).__name__}: {error}")

# 2- refused by the machine. Reading a parameter the master does not know is read-only.
try:
  await star.driver.send_command(module="C0", command="RA", ra="zz")
  print("machine     : no error - the master accepted an unknown parameter name")
except Exception as error:  # noqa: BLE001 - what it raises is exactly what is being checked
  print(f"machine     : {type(error).__name__}: {error}")
  errors = getattr(error, "errors", None)
  if errors:
    for module, decoded in errors.items():
      print(f"  {module}: {decoded}")

## 17- What the rig settled, and what the driver does about it

These were open questions; the sections that answered them are gone, and what they found is in the
driver. Kept here so nobody re-derives them.

**An X move always arrives.** Every condition, every repeat, ends exactly on target. What looked
like a 0.2 to 0.4 mm error was a position read taken before the arm had stopped: the move's reply
comes when the move ends. `move_x` now reads until two reads in a row find the arm at the target.

**How the arm gets there depends on the acceleration.** At index 3 it approaches and never
overshoots; at index 5 it swings past and comes back, in five runs of five. A 96-head parked
forward makes the swing bigger - 0.42 against 0.32 mm - which is the arm carrying an off-centre
mass. It all settles in 27 to 90 ms.

**The master and the X-drive board agree** on where the arm is, so reading `X0 RX` is equivalent to
reading `C0 RX`, and neither `X0 XP` nor `C0 JX` closes a position loop the other does not.

**The 96-head's permitted Y area starts at 102.000 mm**, not the 93.75 mm its own parameter accepts.
Found by bisection, and it is exact: 6528 increments, to the increment. Nothing stores it - all 499
readable parameters across the head and the master were swept and none holds it - so it is compiled
into the firmware and the driver carries it as a measured constant. What it costs: on the frontmost
site of a standard tip carrier the head's A1 reaches rows A to E, and misses F by 1.2 mm.

**`C0 QA` never answers 0, and rounds to the nearest track.** Moved to track 10 it answers 10;
halfway to track 11 it answers 11, not 0 and not 10; and walking 2 to 10 mm past track 10 it still
answers 10. So a caller may not read 0 as "between tracks" - the value is always a track, and the
one it is nearest to. The simulator cannot reproduce this: it answers from the track it was last
sent to.

**The 96-head's Y drive settles within about 0.1 mm.** `H0 RY` answers two counters, the firmware's
and the hardware's, and the driver reads the hardware's. Commanded 6528 increments it reports 6534,
and commanded 35485 it reports 35491 then 35482 - six increments either way, 0.09 mm. The X drive
arrives exactly; this one does not, so anything comparing a commanded Y against the read-back has to
allow for it.

**The autoload's scanner resolves 0.1 mm per increment**, measured as exactly 22.5 mm per track
between tracks 10 and 20, and confirmed by the unit's own configuration, which discovery now reads
rather than assuming. Its drive counts from track 1, a hundred millimetres along the deck: at track
10 it reads 202.5 mm, exactly nine tracks, so its zero sits on track 1 itself rather than half a
track off it.

**The front cover is watched by an input, not by `C0 QC`.** With the monitoring bit clear, `C0 RW`
char 1 followed the cover - 1 shut, 0 open - while `QC` answered "closed" both times. Whether `QC`
works on a machine that declares the monitoring is still open, and needs the configuration write in
section 18.

## 18- The instrument configuration, and what writing it would mean

`C0 AK` writes the machine's non-volatile configuration. It takes **21 parameters**, each with a
default, and the master's convention is that an unsent parameter takes its default - so a partial
`AK` does not change one field, it rewrites all of them. Sending `kb` alone would declare no
channels, no 96-head, a different arm width and a different waste position.

Every one of the 21 is readable: `C0 RM` answers `kb` and `kp`, `C0 QM` the other 19. The cell below
reads them, rebuilds the command that would restore exactly what the machine says today, and shows
what changes if the front cover monitoring bit is set. **It sends nothing.**

Why we would want to: with `kb` bit 2 clear, `C0 QC` answered `qc1` with the cover open, so the
master is not reading the switch. Setting the bit is the only way to find out whether `QC` reports
the cover on a machine that declares the monitoring - and it is also what makes the machine abort a
run when the cover opens, which is why it was turned off in the first place.


In [ ]:
import re

# The 21 parameters AK takes, in the order the specification lists them.
AK_PARAMETERS = "ka ke xt xa xw kb xl xn xr xo xm xx xu xv kp ys kl km ym yu yx".split()


def read_fields(reply: str) -> dict:
  """The two-letter fields in a reply, as the machine wrote them."""
  return dict(re.findall(r"([a-z]{2})([0-9A-Fa-f]+)", reply.split("er00/00", 1)[-1]))


if protocol_mode != "execution":
  raise SystemExit("nothing to read: a simulated machine has no configuration to rebuild")

machine = await star.driver.send_command(module="C0", command="RM")
extended = await star.driver.send_command(module="C0", command="QM")
read = {**read_fields(extended), **read_fields(machine)}

missing = [name for name in AK_PARAMETERS if name not in read]
print(f"read {len(AK_PARAMETERS) - len(missing)} of {len(AK_PARAMETERS)} parameters")
if missing:
  print(f"MISSING, so a safe write is not possible: {missing}")
else:
  as_it_stands = "".join(f"{name}{read[name]}" for name in AK_PARAMETERS)
  print(f"\nrestores exactly what the machine says now:\n  C0AK{as_it_stands}")

  with_monitoring = dict(read)
  with_monitoring["kb"] = f"{int(read['kb'], 16) | 0b100:02X}"
  proposed = "".join(f"{name}{with_monitoring[name]}" for name in AK_PARAMETERS)
  print(f"\nwith the front cover monitoring bit set:\n  C0AK{proposed}")
  print(f"\nkb {read['kb']} -> {with_monitoring['kb']}")
  print(
    "everything else identical:",
    as_it_stands.replace(f"kb{read['kb']}", "")
    == proposed.replace(f"kb{with_monitoring['kb']}", ""),
  )

### Writing it

Only with `allow_configuration_write = True`, set in the cell itself so it cannot be reached by
running the notebook top to bottom. It writes the full command built above, reads the configuration
back, and prints the restore command in case the read-back does not match.

Keep the restore line from the cell above. If anything goes wrong, sending it puts the machine back.


In [ ]:
allow_configuration_write = False

if not allow_configuration_write:
  print("skipped. this rewrites the machine's non-volatile configuration")
elif missing:
  print("refused: not every parameter could be read")
else:
  print(f"restore command, keep this:\n  C0AK{as_it_stands}\n")
  print(await star.driver.send_raw_command(f"C0AK{proposed}"))

  after = {
    **read_fields(await star.driver.send_command(module="C0", command="QM")),
    **read_fields(await star.driver.send_command(module="C0", command="RM")),
  }
  for name in AK_PARAMETERS:
    if after.get(name) != with_monitoring.get(name):
      print(f"  {name}: wrote {with_monitoring.get(name)}, reads back {after.get(name)}")
  print(
    "read back identical to what was written:",
    all(after.get(n) == with_monitoring.get(n) for n in AK_PARAMETERS),
  )

## 19- What the autoload holds in its own memory

Read-only, nothing moves. Four reads the driver did not have until now, each replacing something it
was assuming.

`request_module_configuration` is the one that matters. Its first field is the scanner's step size -
0.1 or 0.125 mm depending on the unit - which the driver used to hardcode; its second says whether
the loading indicators are fitted. Discovery reads both now, so this section checks that the read
works on hardware and that it agrees with what this machine was assumed to be. Last run answered
`au0 0 0 0 0`, so: 0.1 mm per step, indicators fitted.

The other three are diagnostic. `request_adjustment_status` says whether this autoload has ever been
adjusted - an unadjusted module holds factory defaults rather than its own values, and nothing
derived from them means much. `request_init_slot` gives the track the X drive homes against.
`request_adjustment_values` returns the whole adjustment block unparsed, because the 96-head's
equivalent came back truncated and the shape is worth seeing before anyone writes a parser.

`request_parameter` reads any of the 56 named parameters the module stores. A few worth having are
below; the full sweep belongs in the capture script, not here.


In [ ]:
autoload = star.driver.autoload
if autoload is None:
  print("no autoload on this machine")
else:
  c = autoload.configuration

  # What discovery already read off this unit, and what the driver would have assumed without it.
  step, indicators = await autoload.request_module_configuration()
  print(f"scanner step      : {step} mm  (discovery stored {c.x_drive_mm_per_increment})")
  print(f"loading indicators: {'fitted' if indicators else 'none'}")
  if step != 0.1:
    print("  ! this unit is NOT the 0.1 mm generation - every autoload distance depended on that")

  adjusted_on, adjusted = await autoload.request_adjustment_status()
  print(f"adjusted          : {adjusted} on {adjusted_on}")
  if not adjusted:
    print("  ! unadjusted: its stored values are factory defaults, not this unit's")

  print(f"X drive homes at  : track {await autoload.request_init_slot()}")
  print(f"adjustment block  : {await autoload.request_adjustment_values()}")

  # Named parameters worth having: each drive's stored initialization position, and the barcode
  # reading geometry the carrier loads use.
  for name in ("kx", "ky", "kz", "bi", "bp", "bw", "cn", "co", "yl"):
    try:
      print(f"  {name} -> {await autoload.request_parameter(name)}")
    except Exception as e:  # noqa: BLE001 - a name this firmware does not know is an answer too
      print(f"  {name} -> refused: {e}")

## 20- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [ ]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

## 21- Teardown


In [ ]:
await star.stop()
print("disconnected. connected:", star.driver.connected)

# The log is append-only and stays open for the rest of the session - nothing to close.